# HINT vs custom dataset — head-to-head

Edit `USER_CSV` below to point at your own CSV (HINT 10-column format). Run all cells.

Run from the repo root so `data/phase_*_*.csv` resolves.

In [1]:
import ast, pickle, re
import pandas as pd

#USER_CSV = "../../drug-success-report/docs/features/hint_dataset.csv"   # <-- change me
USER_CSV = "../../drug-success-report/outputs/features/hint_dataset.csv"   # <-- change me
PHASES = ["I", "II", "III"]

user = pd.read_csv(USER_CSV)
hint = pd.concat(
    [pd.read_csv(f"data/phase_{p}_{s}.csv").assign(_split=f"{p}_{s}") for p in PHASES for s in ["train", "valid", "test"]],
    ignore_index=True,
)

# normalize phase string so HINT ("phase 1") and user ("Phase 1") line up
def norm_phase(v):
    s = str(v).strip().lower().replace("phase", "").strip()
    return {"1": "I", "i": "I", "2": "II", "ii": "II", "3": "III", "iii": "III"}.get(s)

user["_phase"] = user["phase"].map(norm_phase)
hint["_phase"] = hint["phase"].map(norm_phase)

print(f"user rows: {len(user):>6}    HINT rows: {len(hint):>6}")

user rows:  15318    HINT rows:  12477


In [2]:
import random
nct_id_overlap = set(user["nctid"]).intersection(set(hint["nctid"]))
sample = random.sample(list(nct_id_overlap), 1)

In [3]:
combined = user.join(hint.set_index("nctid"), on="nctid", lsuffix="_user", rsuffix="_hint", how="inner")
combined[combined["label_user"] != combined["label_hint"]]

,nctid,status_user,why_stop_user,label_user,phase_user,diseases_user,icdcodes_user,drugs_user,smiless_user,criteria_user,...,why_stop_hint,label_hint,phase_hint,diseases_hint,icdcodes_hint,drugs_hint,smiless_hint,criteria_hint,_split,_phase_hint
17,NCT01072929,Completed,NaN,0,Phase 3,['depression'],"['F32.A', 'F53.0', 'P91.4', 'Z13.31', 'Z13.32']",['armodafinil'],['NC(=O)C[S@@](=O)C(C1=CC=CC=C1)C1=CC=CC=C1'],Inclusion Criteria:\n\n* The patient has a dia...,...,NaN,1,phase 3,['depression'],"[""['F32.A', 'F53.0', 'P91.4', 'Z13.31', 'Z13.3...","['armodafinil', 'placebo']","['NC(=O)CS(=O)C(C1=CC=CC=C1)C1=CC=CC=C1', 'CN1...",\n Inclusion Criteria:\n\n - ...,III_train,III
40,NCT02666664,Completed,NaN,0,Phase 3,['hypercholesterolemia'],"['I67.2', 'I70.90', 'I70.91', 'I70.0', 'I70.1'...",['etc-1002'],['CC(C)(CCCCCC(O)CCCCCC(C)(C)C(=O)O)C(=O)O'],Inclusion Criteria:\n\n* Fasting LDL-C ≥ 70 mg...,...,NaN,1,phase 3,"['hypercholesterolemia', 'atherosclerotic card...","[""['E78.01', 'E78.00', 'Z83.42']"", ""['Q93.81',...","['etc-1002', 'placebo']","['CC(C)(CCCCCC(O)CCCCCC(C)(C)C(O)=O)C(O)=O', '...",\n Inclusion Criteria:\n\n - ...,III_test,III
56,NCT01094886,Completed,NaN,0,Phase 3,['arthritis'],"['A01.04', 'A02.23', 'A39.83', 'A39.84', 'A54....",['rivaroxaban'],['ClC1=CC=C(S1)C(=O)NC[C@H]1CN(C(=O)O1)C1=CC=C...,Inclusion Criteria:\n\n* Undergone elective to...,...,NaN,1,phase 3,"['arthritis', 'osteoarthritis, knee', 'osteoar...","[""['A01.04', 'A02.23', 'A39.83', 'A39.84', 'A5...",['rivaroxaban'],['ClC1=CC=C(S1)C(=O)NC[C@H]1CN(C(=O)O1)C1=CC=C...,\n Inclusion Criteria:\n\n - ...,III_train,III
68,NCT00737932,Completed,NaN,0,Phase 2,"[""crohn's disease""]","['K50.90', 'K50.913', 'K50.914', 'K50.911', 'K...",['laquinimod'],['CCN(C(=O)C1=C(O)C2=C(C=CC=C2Cl)N(C)C1=O)C1=C...,Inclusion Criteria:\n\n1. Subjects diagnosed w...,...,NaN,1,phase 2,"[""crohn's disease""]","[""['K50.90', 'K50.913', 'K50.914', 'K50.911', ...",['laquinimod'],['CCN(C(=O)C1=C(O)C2=C(C=CC=C2Cl)N(C)C1=O)C1=C...,\n Inclusion Criteria:\n\n 1. ...,II_train,II
130,NCT02207088,Completed,NaN,0,Phase 3,['chronic hepatitis c'],"['K74.01', 'K74.02', 'K68.2', 'E84.9', 'J84.10...",['dasabuvir'],['COC1=C(C=C(C=C1C1=CC2=CC=C(NS(C)(=O)=O)C=C2C...,Inclusion Criteria:\n\n1. Positive for anti-HC...,...,NaN,1,phase 3,"['chronic hepatitis c', 'hepatitis c virus', '...","[""['B18.2', 'B18.0', 'B18.1', 'B18.8', 'B18.9'...","['ombitasvir/paritaprevir/ritonavir', 'dasabuv...",['[H][C@@]12C[C@]1(NC(=O)[C@]1([H])C[C@H](CN1C...,\n Inclusion Criteria:\n\n 1. ...,III_test,III
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15235,NCT01103414,Completed,NaN,0,Phase 2,['type 2 diabetes'],"['P70.2', 'O24.92', 'Z83.3', 'E10.65', 'E10.9'...",['mitoglitazone'],['CCC1=CN=C(C=C1)C(=O)COC1=CC=C(CC2SC(=O)NC2=O...,Inclusion Criteria\n\n1. Males and females wit...,...,NaN,1,phase 2,['type 2 diabetes'],"[""['E11.65', 'E11.9', 'E11.21', 'E11.36', 'E11...","['mitoglitazone', 'mitoglitazone', 'mitoglitaz...",['CCC1=CN=C(C=C1)C(=O)COC1=CC=C(CC2SC(=O)NC2=O...,\n Inclusion Criteria\n\n 1. M...,II_train,II
15260,NCT01041859,Completed,NaN,0,Phase 3,['diabetic peripheral neuropathy'],"['N50.82', 'R07.2', 'R07.82', 'R10.13', 'R10.2...",['tapentadol extended release (er)'],['CC[C@H]([C@@H](C)CN(C)C)C1=CC(O)=CC=C1'],Inclusion Criteria:\n\n* Patients with Type 1 ...,...,NaN,1,phase 3,['diabetic peripheral neuropathy'],"[""['A18.2', 'H11.043', 'H11.053', 'H18.463', '...","['tapentadol extended release (er)', 'placebo']","['CC[C@H]([C@@H](C)CN(C)C)C1=CC(O)=CC=C1', 'CN...",\n Inclusion Criteria:\n\n - ...,III_train,III
15264,NCT02186834,Completed,NaN,1,Unknown,['multiple myeloma'],"['C90.01', 'C90.02', 'C90.00']",['selinexor'],['FC(F)(F)C1=CC(=CC(=C1)C1=NN(\\C=C/C(=O)NNC2=...,Inclusion Criteria:\n\n* Patients with relapse...,...,NaN,0,phase 1/phase 2,['multiple myeloma'],"[""['C90.01', 'C90.02', 'C90.00']""]","['selinexor', 'liposomal doxorubicin', 'dexame...",['FC(F)(F)C1=CC(=CC(=C1)C1=NN(\\C=C

In [4]:
import os, sys, tempfile
import numpy as np, torch
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, accuracy_score,
)
sys.path.append("."); sys.path.append("HINT")
from HINT.dataloader import csv_three_feature_2_dataloader
from run_inference import write_phase_csv

PHASE_TO_CKPT = {
    "I":   "save_model/phase_I.ckpt",
    "II":  "save_model/phase_II.ckpt",
    "III": "save_model/phase_III.ckpt",
}

# Pair shared rows: user features keyed by nctid; HINT label + HINT phase + HINT features by nctid
user_indexed = user.set_index("nctid")
hint_indexed = hint.drop_duplicates("nctid").set_index("nctid")
#shared_ids = sorted(set(user_indexed.index) & set(hint_indexed.index))
combined = user.join(hint.set_index("nctid"), on="nctid", lsuffix="_user", rsuffix="_hint", how="inner")
shared_ids = combined[
    (combined["diseases_user"] == combined["diseases_hint"])
    #& (combined["drugs_user"] == combined["drugs_hint"])
].nctid
shared_ids = [i for i in shared_ids if hint_indexed.loc[i, "_phase"] in PHASE_TO_CKPT]
print(f"shared rows scored: {len(shared_ids)}")

def score_features(feature_df, hint_label_series, phase, ckpt):
    fd, tmp = tempfile.mkstemp(suffix=".csv"); os.close(fd)
    write_phase_csv(feature_df, tmp)
    loader = csv_three_feature_2_dataloader(tmp, shuffle=False, batch_size=32)
    model = torch.load(ckpt, weights_only=False, map_location="cpu").to("cpu")
    if hasattr(model, "set_device"):
        model.set_device(torch.device("cpu"))
    model.eval()
    with torch.no_grad():
        _, preds, _, _ = model.generate_predict(loader)
    os.unlink(tmp)
    return np.asarray(preds, dtype=float)

def point_metrics(y, p, thr=0.5):
    yhat = (p > thr).astype(int)
    return {
        "n": len(y), "pos_rate": float(y.mean()),
        "ROC-AUC":   roc_auc_score(y, p) if len(set(y)) > 1 else float("nan"),
        "PR-AUC":    average_precision_score(y, p) if len(set(y)) > 1 else float("nan"),
        "F1":        f1_score(y, yhat),
        "Precision": precision_score(y, yhat, zero_division=0),
        "Recall":    recall_score(y, yhat, zero_division=0),
        "Accuracy":  accuracy_score(y, yhat),
    }

def bootstrap_metrics(y, p, n_boot=20, thr=0.5):
    rng = np.random.default_rng(0)
    n = len(y); rows = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yi, pi = y[idx], p[idx]
        if len(set(yi)) < 2: continue
        rows.append([
            roc_auc_score(yi, pi),
            average_precision_score(yi, pi),
            f1_score(yi, (pi > thr).astype(int)),
        ])
    if not rows: return {}
    a = np.array(rows)
    return {
        "ROC-AUC mean": float(a[:,0].mean()), "ROC-AUC std": float(a[:,0].std()),
        "PR-AUC mean":  float(a[:,1].mean()), "PR-AUC std":  float(a[:,1].std()),
        "F1 mean":      float(a[:,2].mean()), "F1 std":      float(a[:,2].std()),
    }

all_user_preds = []
all_hint_preds = []
all_labels = []
rows = []
for phase in ["I", "II", "III"]:
    ids_p = [i for i in shared_ids if hint_indexed.loc[i, "_phase"] == phase]
    if not ids_p:
        print(f"phase {phase}: 0 shared rows, skipping"); continue

    user_features = user_indexed.loc[ids_p].reset_index()
    hint_features = hint_indexed.loc[ids_p].reset_index()
    y_hint = hint_indexed.loc[ids_p, "label"].astype(int).to_numpy()

    p_user = score_features(user_features, y_hint, phase, PHASE_TO_CKPT[phase])
    p_hint = score_features(hint_features, y_hint, phase, PHASE_TO_CKPT[phase])
    ind = 1
    #print(user_features.iloc[ind])
    #print(hint_features.iloc[ind])
    #print(user_features.iloc[ind].smiless)
    #print(hint_features.iloc[ind].smiless)
    #print(user_features.iloc[ind].icdcodes)
    #print(hint_features.iloc[ind].icdcodes)
    #print(user_features.iloc[ind].criteria)
    #print(hint_features.iloc[ind].criteria)
    #print(p_user)
    #print(p_hint)
    #print(y_hint)
    all_user_preds.extend(p_user)
    all_hint_preds.extend(p_hint)
    all_labels.extend(y_hint)

    rows.append({"phase": phase, "features": "user", **point_metrics(y_hint, p_user), **bootstrap_metrics(y_hint, p_user)})
    rows.append({"phase": phase, "features": "HINT", **point_metrics(y_hint, p_hint), **bootstrap_metrics(y_hint, p_hint)})

metrics_df = pd.DataFrame(rows).set_index(["phase", "features"])
metrics_df.round(3)

shared rows scored: 722


n  pos_rate  ROC-AUC  PR-AUC     F1  Precision  Recall  \
phase features                                                             
I     user       72     0.639    0.599   0.679  0.705      0.738   0.674   
      HINT       72     0.639    0.692   0.803  0.784      0.714   0.870   
II    user      404     0.550    0.635   0.687  0.686      0.611   0.784   
      HINT      404     0.550    0.694   0.723  0.729      0.646   0.838   
III   user      246     0.740    0.683   0.835  0.832      0.798   0.868   
      HINT      246     0.740    0.760   0.876  0.859      0.820   0.901   

                Accuracy  ROC-AUC mean  ROC-AUC std  PR-AUC mean  PR-AUC std  \
phase features                                                                 
I     user         0.639         0.610        0.066        0.701       0.039   
      HINT         0.694         0.690        0.070        0.811       0.043   
II    user         0.606         0.631        0.029        0.689       0.030   
      HINT         0.658         0.693        0.027        0.729       0.028   
III   user         0.740         0.688        0.048        0.836       0.033   
      HINT         0.780         0.764        0.041        0.878       0.032   

                F1 mean  F1 std  
phase features                   
I     user        0.713   0.048  
      HINT        0.782   0.035  
II    user        0.687   0.023  
      HINT        0.733   0.019  
III   user        0.831   0.024  
      HINT        0.855   0.016

In [25]:
agree = combined[combined["label_user"] == combined["label_hint"]]

## 1. Phase distribution

In [26]:
phase_counts = pd.DataFrame(
    {
        "user": user["_phase"].value_counts(dropna=False),
        "HINT": hint["_phase"].value_counts(dropna=False),
    }
).fillna(0).astype(int)
phase_counts

,user,HINT
_phase,,
II,5343,5229
III,3496,4324
None,3456,1799
I,3023,1125


## 2. Positive-label rate per phase

HINT's TOP label is "completed and met primary endpoint." If your positive rate diverges sharply, you're probably encoding a different outcome (e.g. regulatory approval). The model was trained against HINT's definition and cannot recover yours from the same inputs.

In [27]:
rows = []
for p in PHASES + [None]:
    u = user if p is None else user[user["_phase"] == p]
    h = hint if p is None else hint[hint["_phase"] == p]
    rows.append({
        "phase": p or "ALL",
        "user_n": len(u), "user_pos_rate": round(u["label"].mean(), 3) if len(u) else None,
        "HINT_n": len(h), "HINT_pos_rate": round(h["label"].mean(), 3) if len(h) else None,
    })
pd.DataFrame(rows)

,phase,user_n,user_pos_rate,HINT_n,HINT_pos_rate
0,I,3023,0.323,1125,0.580
1,II,5343,0.273,5229,0.492
2,III,3496,0.352,4324,0.689
3,ALL,15318,0.350,12477,0.573


## 3. Status × label cross-tab

How does `Completed` map to `label=1` in each dataset? In HINT data, completed trials are positives ~75-86% of the time. If yours are flipped, the labeling definitions don't match.

In [28]:
def status_label_rate(df):
    df = df.copy()
    df["status"] = df["status"].astype(str).str.lower().str.strip()
    g = df.groupby("status")["label"].agg(["size", "mean"]).rename(columns={"size": "n", "mean": "pos_rate"})
    return g.round(3).sort_values("n", ascending=False)

print("USER:"); display(status_label_rate(user))
print("HINT:"); display(status_label_rate(hint))

USER:


,n,pos_rate
status,,
completed,8011,0.444
terminated,2998,0.000
unknown,1667,0.505
withdrawn,1513,0.000
recruiting,685,0.928
"active, not recruiting",339,0.959
suspended,105,0.000


HINT:


,n,pos_rate
status,,
completed,8978,0.725
terminated,2470,0.036
"active, not recruiting",377,0.793
unknown status,362,0.652
withdrawn,252,0.008
recruiting,22,0.818
suspended,15,0.067
not yet recruiting,1,0.000


## 4. NCT-ID overlap and label agreement on the overlap

If a substantial number of NCT IDs appear in both datasets, we can compare labels directly. Disagreement here pins the issue on labeling conventions, not on features or the model.

In [29]:
user_ids = set(user["nctid"])
hint_ids = set(hint["nctid"])
shared = user_ids & hint_ids
print(f"shared NCT IDs: {len(shared)} / user {len(user_ids)} ({100*len(shared)/max(1,len(user_ids)):.1f}%)")

if shared:
    hmap = dict(zip(hint["nctid"], hint["label"]))
    sub = user[user["nctid"].isin(shared)].copy()
    sub["hint_label"] = sub["nctid"].map(hmap).astype(int)
    print(f"label agreement on shared rows: {(sub['label']==sub['hint_label']).mean():.3f}")
    print()
    print("Per-phase agreement:")
    for p in PHASES:
        s = sub[sub["_phase"] == p]
        if len(s):
            print(f"  Phase {p:<3} n={len(s):>4}  agreement={(s['label']==s['hint_label']).mean():.3f}")
    print()
    print("Confusion (rows = user, cols = HINT):")
    display(pd.crosstab(sub["label"], sub["hint_label"], rownames=["user"], colnames=["HINT"]))

shared NCT IDs: 1235 / user 15318 (8.1%)
label agreement on shared rows: 0.678

Per-phase agreement:
  Phase I   n= 146  agreement=0.630
  Phase II  n= 617  agreement=0.652
  Phase III n= 406  agreement=0.692

Confusion (rows = user, cols = HINT):


HINT,0,1
user,,
0,457,356
1,42,380


## 5. ICD-10 code coverage

How many user codes are recognized by `data/icdcode2ancestor_dict.pkl`? Codes not in the dict are silently dropped by `GRAM.forward_code_lst`, so missing coverage = lost disease signal.

In [30]:
with open("data/icdcode2ancestor_dict.pkl", "rb") as f:
    icd2anc = pickle.load(f)

def parse_user_codes(cell):
    s = str(cell)
    if not s or s == "nan": return []
    try:
        return ast.literal_eval(s)
    except Exception:
        return re.findall(r"'([^']+)'", s)

def parse_hint_codes(cell):
    s = str(cell); out = []
    try:
        for inner in ast.literal_eval(s):
            out.extend(ast.literal_eval(inner) if isinstance(inner, str) else inner)
    except Exception:
        pass
    return out

def coverage(codes):
    codes = list(codes)
    if not codes: return None
    uniq = set(codes)
    return {
        "mentions": len(codes),
        "unique": len(uniq),
        "mention_hit_rate": round(sum(c in icd2anc for c in codes) / len(codes), 3),
        "unique_hit_rate": round(sum(c in icd2anc for c in uniq) / len(uniq), 3),
    }

user_codes = [c for cell in user["icdcodes"] for c in parse_user_codes(cell)]
hint_codes = [c for cell in hint["icdcodes"] for c in parse_hint_codes(cell)]
pd.DataFrame({"user": coverage(user_codes), "HINT": coverage(hint_codes)})

,user,HINT
mentions,76512.000,102770.0
unique,5529.000,3680.0
mention_hit_rate,0.722,1.0
unique_hit_rate,0.473,1.0


## 6. Drug overlap

How many of the user's drugs were seen during HINT training? Out-of-distribution molecules force the MPNN to extrapolate.

In [31]:
def parse_drugs(cell):
    s = str(cell)
    if not s or s == "nan": return []
    try:
        return [d.strip().lower() for d in ast.literal_eval(s)]
    except Exception:
        return [d.strip().lower() for d in re.findall(r"'([^']+)'", s)]

user_drugs = {d for cell in user["drugs"] for d in parse_drugs(cell)}
hint_drugs = {d for cell in hint["drugs"] for d in parse_drugs(cell)}
shared_drugs = user_drugs & hint_drugs
print(f"unique user drugs: {len(user_drugs)}")
print(f"unique HINT drugs: {len(hint_drugs)}")
print(f"shared:            {len(shared_drugs)} ({100*len(shared_drugs)/max(1,len(user_drugs)):.1f}% of user)")

unique user drugs: 5023
unique HINT drugs: 10030
shared:            1463 (29.1% of user)


## 7. Sentence-embedding coverage (BioBERT branch)

If `data/sentence2embedding.pkl` is empty, the protocol encoder is silent for everyone — both datasets — and the model is running with 2 of 3 encoders.

In [32]:
with open("data/sentence2embedding.pkl", "rb") as f:
    s2e = pickle.load(f)
print(f"sentence2embedding entries: {len(s2e)}")
if len(s2e) == 0:
    print("WARNING: protocol2feature returns zeros for every trial. BioBERT branch is dead.")

sentence2embedding entries: 0


## 8. Run HINT models on the shared rows — user features vs HINT features, both judged against HINT labels

Restrict to NCT IDs in **both** tables. For each shared trial, run the appropriate phase model **twice** — once on the user CSV's feature row, once on the HINT CSV's feature row — and score both prediction sets against the **HINT label**. Phase is taken from HINT (so the model and the label agree).

What this isolates: identical model + identical labels + identical NCT IDs. The only thing that differs is the feature payload (SMILES list, ICD-code list, criteria text, drug-name list). Any gap is attributable to **how each dataset encodes the same trial**, not to labels, label noise, or model differences.

In [ ]:
import os, sys, tempfile
import numpy as np, torch
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, accuracy_score,
)
sys.path.append("."); sys.path.append("HINT")
from HINT.dataloader import csv_three_feature_2_dataloader
from run_inference import write_phase_csv

PHASE_TO_CKPT = {
    "I":   "save_model/phase_I.ckpt",
    "II":  "save_model/phase_II.ckpt",
    "III": "save_model/phase_III.ckpt",
}

# Pair shared rows: user features keyed by nctid; HINT label + HINT phase + HINT features by nctid
user_indexed = user.set_index("nctid")
hint_indexed = hint.drop_duplicates("nctid").set_index("nctid")
#shared_ids = sorted(set(user_indexed.index) & set(hint_indexed.index))
combined = user.join(hint.set_index("nctid"), on="nctid", lsuffix="_user", rsuffix="_hint", how="inner")
shared_ids = combined[
    (combined["diseases_user"] == combined["diseases_hint"])
    #& (combined["drugs_user"] == combined["drugs_hint"])
].nctid
shared_ids = [i for i in shared_ids if hint_indexed.loc[i, "_phase"] in PHASE_TO_CKPT]
print(f"shared rows scored: {len(shared_ids)}")

def score_features(feature_df, hint_label_series, phase, ckpt):
    fd, tmp = tempfile.mkstemp(suffix=".csv"); os.close(fd)
    write_phase_csv(feature_df, tmp)
    loader = csv_three_feature_2_dataloader(tmp, shuffle=False, batch_size=32)
    model = torch.load(ckpt, weights_only=False, map_location="cpu").to("cpu")
    if hasattr(model, "set_device"):
        model.set_device(torch.device("cpu"))
    model.eval()
    with torch.no_grad():
        _, preds, _, _ = model.generate_predict(loader)
    os.unlink(tmp)
    return np.asarray(preds, dtype=float)

def point_metrics(y, p, thr=0.5):
    yhat = (p > thr).astype(int)
    return {
        "n": len(y), "pos_rate": float(y.mean()),
        "ROC-AUC":   roc_auc_score(y, p) if len(set(y)) > 1 else float("nan"),
        "PR-AUC":    average_precision_score(y, p) if len(set(y)) > 1 else float("nan"),
        "F1":        f1_score(y, yhat),
        "Precision": precision_score(y, yhat, zero_division=0),
        "Recall":    recall_score(y, yhat, zero_division=0),
        "Accuracy":  accuracy_score(y, yhat),
    }

def bootstrap_metrics(y, p, n_boot=20, thr=0.5):
    rng = np.random.default_rng(0)
    n = len(y); rows = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yi, pi = y[idx], p[idx]
        if len(set(yi)) < 2: continue
        rows.append([
            roc_auc_score(yi, pi),
            average_precision_score(yi, pi),
            f1_score(yi, (pi > thr).astype(int)),
        ])
    if not rows: return {}
    a = np.array(rows)
    return {
        "ROC-AUC mean": float(a[:,0].mean()), "ROC-AUC std": float(a[:,0].std()),
        "PR-AUC mean":  float(a[:,1].mean()), "PR-AUC std":  float(a[:,1].std()),
        "F1 mean":      float(a[:,2].mean()), "F1 std":      float(a[:,2].std()),
    }

all_user_preds = []
all_hint_preds = []
all_labels = []
rows = []
for phase in ["I", "II", "III"]:
    ids_p = [i for i in shared_ids if hint_indexed.loc[i, "_phase"] == phase]
    if not ids_p:
        print(f"phase {phase}: 0 shared rows, skipping"); continue

    user_features = user_indexed.loc[ids_p].reset_index()
    hint_features = hint_indexed.loc[ids_p].reset_index()
    y_hint = hint_indexed.loc[ids_p, "label"].astype(int).to_numpy()

    p_user = score_features(user_features, y_hint, phase, PHASE_TO_CKPT[phase])
    p_hint = score_features(hint_features, y_hint, phase, PHASE_TO_CKPT[phase])
    ind = 1
    #print(user_features.iloc[ind])
    #print(hint_features.iloc[ind])
    #print(user_features.iloc[ind].smiless)
    #print(hint_features.iloc[ind].smiless)
    #print(user_features.iloc[ind].icdcodes)
    #print(hint_features.iloc[ind].icdcodes)
    #print(user_features.iloc[ind].criteria)
    #print(hint_features.iloc[ind].criteria)
    #print(p_user)
    #print(p_hint)
    #print(y_hint)
    all_user_preds.extend(p_user)
    all_hint_preds.extend(p_hint)
    all_labels.extend(y_hint)

    rows.append({"phase": phase, "features": "user", **point_metrics(y_hint, p_user), **bootstrap_metrics(y_hint, p_user)})
    rows.append({"phase": phase, "features": "HINT", **point_metrics(y_hint, p_hint), **bootstrap_metrics(y_hint, p_hint)})

metrics_df = pd.DataFrame(rows).set_index(["phase", "features"])
metrics_df.round(3)

shared rows scored: 722


n  pos_rate  ROC-AUC  PR-AUC     F1  Precision  Recall  \
phase features                                                             
I     user       72     0.639    0.599   0.679  0.705      0.738   0.674   
      HINT       72     0.639    0.692   0.803  0.784      0.714   0.870   
II    user      404     0.550    0.635   0.687  0.686      0.611   0.784   
      HINT      404     0.550    0.694   0.723  0.729      0.646   0.838   
III   user      246     0.740    0.683   0.835  0.832      0.798   0.868   
      HINT      246     0.740    0.760   0.876  0.859      0.820   0.901   

                Accuracy  ROC-AUC mean  ROC-AUC std  PR-AUC mean  PR-AUC std  \
phase features                                                                 
I     user         0.639         0.610        0.066        0.701       0.039   
      HINT         0.694         0.690        0.070        0.811       0.043   
II    user         0.606         0.631        0.029        0.689       0.030   
      HINT         0.658         0.693        0.027        0.729       0.028   
III   user         0.740         0.688        0.048        0.836       0.033   
      HINT         0.780         0.764        0.041        0.878       0.032   

                F1 mean  F1 std  
phase features                   
I     user        0.713   0.048  
      HINT        0.782   0.035  
II    user        0.687   0.023  
      HINT        0.733   0.019  
III   user        0.831   0.024  
      HINT        0.855   0.016

In [14]:
all_rows = [
    {"features": "user", **point_metrics(y_hint, p_user), **bootstrap_metrics(y_hint, p_user)},
    {"features": "HINT", **point_metrics(y_hint, p_hint), **bootstrap_metrics(y_hint, p_hint)},
]
all_df = pd.DataFrame(all_rows).set_index(["features"]).round(3)
all_df

,n,pos_rate,ROC-AUC,PR-AUC,F1,Precision,Recall,Accuracy,ROC-AUC mean,ROC-AUC std,PR-AUC mean,PR-AUC std,F1 mean,F1 std
features,,,,,,,,,,,,,,
user,226,0.677,0.777,0.886,0.842,0.750,0.961,0.757,0.780,0.030,0.885,0.015,0.839,0.015
HINT,226,0.677,0.788,0.886,0.853,0.775,0.948,0.779,0.791,0.035,0.886,0.018,0.848,0.018


In [15]:
metrics = ['ROC-AUC', 'PR-AUC', 'F1', 'Precision', 'Recall', 'Accuracy']
pct_diff = (all_df.loc['user', metrics] - all_df.loc['HINT', metrics]) / all_df.loc['HINT', metrics] * 100
print(pct_diff)

ROC-AUC     -1.395939
PR-AUC       0.000000
F1          -1.289566
Precision   -3.225806
Recall       1.371308
Accuracy    -2.824134
dtype: float64


In [16]:
(np.array(all_user_preds)-np.array(all_hint_preds)).mean()

0.013601748151194023

In [17]:
for i in range(len(shared_ids)):
    print(combined[combined["nctid"] == shared_ids[i]].iloc[0])

nctid                                                  NCT00389870
status_user                                              Completed
why_stop_user                                                  NaN
label_user                                                       0
phase_user                                                 Phase 3
diseases_user                                ['colorectal cancer']
icdcodes_user                                          ['Z15.060']
drugs_user                                        ['cyclosporine']
smiless_user     ['CC[C@@H]1NC(=O)[C@H]([C@H](O)[C@H](C)C\\C=C\...
criteria_user    DISEASE CHARACTERISTICS:\n\n* Diagnosis of col...
_phase_user                                                    III
status_hint                                         unknown status
why_stop_hint                                                  NaN
label_hint                                                       0
phase_hint                                                 pha